# Ouroboros — 3D stage: MACE-OFF benchmarks and error propagation (Phase 6)

Runs the geometry benchmarks on 1,000 test molecules with MACE-OFF23 (GPU) and the energy-error propagation for one evaluated run (predicted vs. true SMILES, by error category). Needs `LOCAL_DATA/manifests/test.tsv` (copied from Drive) and a run whose `eval/predictions.jsonl` exists on Drive.

All project code runs in subprocesses (`!python ...`) so the pinned numpy etc. take effect without restarting the kernel. If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess, json
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true
!python --version && nproc && free -g | head -2

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'claude/vigilant-johnson-j4882f'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # data, runs, results live here
REPO_DIR = '/content/ouroboros-ocsr'
LOCAL_DATA = '/content/data/full'  # configs expect shards at /content/data/full/shards

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for sub in ('data', 'runs', 'results', 'real'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q -r requirements-colab.txt
pip install -q --no-deps -e .
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!mkdir -p {LOCAL_DATA}/manifests && cp {DRIVE_ROOT}/data/full/manifests/test.tsv {LOCAL_DATA}/manifests/
MODEL = 'medium'  # MACE-OFF23 size (Academic Software License)
RUN_ID = 'C_n200k_f0_s0'  # the best arm's run (choose after aggregation)
N_RELAX = 200  # molecules for the convergence benchmark (10 conformers each)
PER_CATEGORY = 100  # error-propagation pairs per error category
OUT = f'{DRIVE_ROOT}/results/geometry'
os.makedirs(OUT, exist_ok=True)

In [ ]:
t = time.time()
!python scripts/bench_geometry.py embed --manifest {LOCAL_DATA}/manifests/test.tsv --n 1000
!python scripts/bench_geometry.py enantio --manifest {LOCAL_DATA}/manifests/test.tsv --n 20 --model {MODEL}
!python scripts/bench_geometry.py relax --manifest {LOCAL_DATA}/manifests/test.tsv --n {N_RELAX} --n-conf 10 --model {MODEL}
!cp benchmarks/geometry/*.json {OUT}/
print(f'benchmarks: {(time.time() - t) / 3600:.2f} GPU-h')

In [ ]:
t = time.time()
!python scripts/error_propagation.py --predictions {DRIVE_ROOT}/runs/{RUN_ID}/eval/predictions.jsonl --set rendered_test --out {OUT}/{RUN_ID} --n-conf 10 --mmff-prescreen 3 --model {MODEL} --skip-correct --per-category {PER_CATEGORY}
print(f'error propagation: {(time.time() - t) / 3600:.2f} GPU-h')

In [ ]:
# ---- report back ----
!cat {OUT}/embed.json | head -20; cat {OUT}/relax.json; grep -E 'max_abs|pass' {OUT}/enantiomers.json
!cat {OUT}/{RUN_ID}/summary.json